# Ordered logit

Generate an ordinal outcome and estimate an ordered-logit model with three learned thresholds.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    OrderedChoiceDataset,
    OrderedLogit,
)

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(18)
n_obs = 500
x = rng.normal(size=n_obs)
u = rng.uniform(size=n_obs)
logistic_error = np.log(u) - np.log1p(-u)
latent_utility = 0.90 * x + logistic_error
y = np.digitize(latent_utility, bins=[-1.0, 0.0, 1.0])

data = OrderedChoiceDataset(
    y=torch.as_tensor(y, dtype=torch.long),
    x={"x": torch.as_tensor(x)},
    weights=torch.ones(n_obs),
    categories=[0, 1, 2, 3],
)
print("Category counts:", dict(zip(*np.unique(y, return_counts=True))))


Category counts: {np.int64(0): np.int64(155), np.int64(1): np.int64(107), np.int64(2): np.int64(98), np.int64(3): np.int64(140)}


In [3]:
latent = Beta("B_X", init=0.50) * "x"
thresholds = [
    Beta("TAU_1", init=-0.80),
    Beta("TAU_2", init=0.00),
    Beta("TAU_3", init=0.80),
]
model = OrderedLogit(
    latent,
    thresholds,
    device=device,
    max_iter=60,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,OrderedLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,60
Line search,strong_wolfe
